# Telcovantage — GPU TrOCR Server

Runs TrOCR models on Colab's free GPU (T4) and exposes them via ngrok tunnel.
Your laptop's Flask backend sends cropped images here instead of running
CPU-bound local inference.

**Workflow:**
1. Run all cells below
2. Copy the `REMOTE_TROCR_URL` printed at the end
3. On your laptop: `set REMOTE_TROCR_URL=<url>` then `python server.py`

---
## Cell 1: Install dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok
# ngrok binary (official via PyPI ngrok package)
!pip install -q ngrok

---
## Cell 2: Clone your repo

In [ ]:
import os
REPO_URL = "https://github.com/jonrenzo/Telcovantage-Site-Map-Reader"
REPO_DIR = "/content/Telcovantage-Site-Map-Reader"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

---
## Cell 3: Install project Python dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

---
## Cell 4: Set ngrok auth token

1. Go to https://dashboard.ngrok.com → get your **authtoken**
2. Paste it below between the quotes

In [ ]:
NGROK_AUTH_TOKEN = ""  # ← PASTE YOUR NGROK TOKEN HERE

import ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok auth set.")

---
## Cell 5: Start TrOCR server + ngrok tunnel

This starts the FastAPI server in background, then creates an ngrok tunnel
to port 8000.

In [ ]:
import subprocess, time, sys

# Kill any previous server on port 8000
!kill -9 $(lsof -ti:8000) 2>/dev/null || true

# Start server in background
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "colab.server:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Wait for server to start
time.sleep(3)
print("Server process started on port 8000.")

# Create ngrok tunnel
listener = ngrok.forward(8000, authtoken_from_env=True)
public_url = listener.url()

print(f"\n{'='*60}")
print(f"  ✅  REMOTE_TROCR_URL = {public_url}")
print(f"{'='*60}")
print()
print("Set this on your laptop:")
print(f"  set REMOTE_TROCR_URL={public_url}")
print(f"  python server.py")
print()
print("Test the connection:")
print(f"  curl {public_url}/health")

---
## Cell 6: Keep-alive

Leaves this notebook connected so Colab doesn't recycle the VM.
Let this cell run indefinitely.

In [ ]:
import time
print("Keep-alive running. Press 🟥 Stop to shut down.")
while True:
    time.sleep(120)